

This notebook runs the whole project end to end in one place:
load the raw CSV, clean it, build features, train several models,
tune the best ones, and save the final model.

It does not depend on any other notebook and does not need to be
run in a particular order relative to other files. Just run every
cell from top to bottom.

**Before running:** make sure your raw CSV is at `data/raw/telco_churn.csv`,
with this notebook placed inside the `notebooks/` folder of the project
(same layout as the other notebooks), or adjust `PROJECT_ROOT` below.

In [ ]:
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

print("Imports done. XGBoost available:", HAS_XGBOOST, "| CatBoost available:", HAS_CATBOOST)

## Stage 0 — Paths and constants

If this notebook lives inside the `notebooks/` folder, `PROJECT_ROOT`
below points one level up, to the project root — same as the other
notebooks in this project.

In [ ]:
# Find the project root from the current working directory.
# This keeps the notebook portable when Jupyter starts from the project root
# or from the notebooks folder.
PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "data" / "raw").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate
        break

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"

for folder in (RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RAW_FILENAMES = [
    "telco_churn.csv",
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "telco_churn.csv.csv",  # covers the double-extension case seen earlier
]

CLEANED_DATA_PATH = PROCESSED_DATA_DIR / "cleaned_telco_churn.csv"
MODEL_PATH = MODELS_DIR / "final_churn_model.joblib"
RISK_TABLE_PATH = REPORTS_DIR / "customer_risk_table.csv"

TARGET_COLUMN = "Churn"
RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DATA_DIR)

## Stage 1 — Load the raw data

In [ ]:
def load_raw_data() -> pd.DataFrame:
    for name in RAW_FILENAMES:
        candidate = RAW_DATA_DIR / name
        if candidate.exists():
            print(f"Found: {candidate}")
            return pd.read_csv(candidate)

    expected = ", ".join(RAW_FILENAMES)
    raise FileNotFoundError(
        f"Could not find the raw dataset in {RAW_DATA_DIR}.\n"
        f"Expected one of: {expected}\n"
        "Rename your CSV to 'telco_churn.csv' and place it in that folder."
    )

df_raw = load_raw_data()
df_raw.shape

## Stage 2 — Clean the data

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Remove extra spaces from column names.
    df.columns = [c.strip() for c in df.columns]

    # TotalCharges is stored as text and has a few blank values.
    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # SeniorCitizen is stored as 0/1 but it means Yes/No.
    if "SeniorCitizen" in df.columns:
        df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

    # Missing TotalCharges usually means a brand-new customer (tenure = 0),
    # so we set those to 0 instead of guessing or dropping them.
    if "TotalCharges" in df.columns and "tenure" in df.columns:
        mask = df["TotalCharges"].isna() & (df["tenure"] == 0)
        df.loc[mask, "TotalCharges"] = 0.0

    # Any remaining missing TotalCharges rows are dropped, since we
    # cannot explain them the same safe way.
    if "TotalCharges" in df.columns:
        df = df.dropna(subset=["TotalCharges"])

    # Drop exact duplicate rows and duplicate customer IDs.
    df = df.drop_duplicates()
    if "customerID" in df.columns:
        df = df.drop_duplicates(subset="customerID")

    # Simplify "No internet service" / "No phone service" down to "No",
    # since that information is already captured in another column.
    no_internet_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies",
    ]
    for col in no_internet_cols:
        if col in df.columns:
            df[col] = df[col].replace("No internet service", "No")
    if "MultipleLines" in df.columns:
        df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

    # Turn the Yes/No target into 1/0 for the model.
    df[TARGET_COLUMN] = df[TARGET_COLUMN].map({"Yes": 1, "No": 0})

    return df

df_clean = clean_data(df_raw)
df_clean.to_csv(CLEANED_DATA_PATH, index=False)
print(f"Saved cleaned data to: {CLEANED_DATA_PATH}")
df_clean.shape

## Stage 3 — Feature engineering

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Group tenure into readable ranges. The last edge always covers
    # the true maximum, even if it is below 60, so this never breaks.
    max_tenure = int(df["tenure"].max())
    if max_tenure <= 60:
        bins = [-1, 12, 24, 48, 60]
        labels = ["0-12", "13-24", "25-48", "49-60"]
    else:
        bins = [-1, 12, 24, 48, 60, max_tenure]
        labels = ["0-12", "13-24", "25-48", "49-60", "60+"]
    df["tenure_group"] = pd.cut(df["tenure"], bins=bins, labels=labels)

    # Average monthly spend, avoiding division by zero for new customers.
    safe_tenure = df["tenure"].replace(0, 1)
    df["average_monthly_charge"] = df["TotalCharges"] / safe_tenure

    # How many add-on services the customer has active.
    addon_cols = [
        "OnlineSecurity", "OnlineBackup", "DeviceProtection",
        "TechSupport", "StreamingTV", "StreamingMovies",
    ]
    present_addon_cols = [c for c in addon_cols if c in df.columns]
    df["service_count"] = (df[present_addon_cols] == "Yes").sum(axis=1)

    # Whether the customer has at least one online-related service.
    online_cols = [c for c in ["OnlineSecurity", "OnlineBackup"] if c in df.columns]
    df["has_online_service"] = (
        (df[online_cols] == "Yes").any(axis=1).map({True: "Yes", False: "No"})
    )

    # Month-to-month contracts tend to churn more, so this is a useful flag.
    df["is_month_to_month"] = (
        (df["Contract"] == "Month-to-month").map({True: "Yes", False: "No"})
    )

    return df

df = add_features(df_clean)
df.shape

## Stage 4 — Train / test split

In [ ]:
drop_cols = [TARGET_COLUMN]
if "customerID" in df.columns:
    drop_cols.append("customerID")
X = df.drop(columns=drop_cols)
y = df[TARGET_COLUMN]

numerical_features = [
    "tenure", "MonthlyCharges", "TotalCharges",
    "average_monthly_charge", "service_count",
]
categorical_features = [c for c in X.columns if c not in numerical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape

## Stage 5 — Model definitions and cross-validation

In [ ]:
def build_preprocessor(numerical_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

def get_models():
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, max_depth=10, random_state=RANDOM_STATE
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=200, max_depth=3, random_state=RANDOM_STATE
        ),
    }
    if HAS_XGBOOST:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.1,
            eval_metric="logloss", random_state=RANDOM_STATE,
        )
    if HAS_CATBOOST:
        models["CatBoost"] = CatBoostClassifier(
            iterations=300, depth=6, learning_rate=0.1,
            random_state=RANDOM_STATE, verbose=False,
        )
    return models

models = get_models()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", build_preprocessor(numerical_features, categorical_features)),
        ("model", model),
    ])
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv,
        scoring=["roc_auc", "f1", "recall"],
    )
    cv_results[name] = {
        "roc_auc": scores["test_roc_auc"].mean(),
        "f1": scores["test_f1"].mean(),
        "recall": scores["test_recall"].mean(),
    }

cv_table = pd.DataFrame(cv_results).T.sort_values("roc_auc", ascending=False)
cv_table

## Stage 6 — Fit each model and compare on the test set

In [ ]:
def evaluate_model(name, y_true, y_pred, y_proba):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
    }

test_results = []
fitted_pipelines = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", build_preprocessor(numerical_features, categorical_features)),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    fitted_pipelines[name] = pipeline

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    test_results.append(evaluate_model(name, y_test, y_pred, y_proba))

comparison_table = pd.DataFrame(test_results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
comparison_table

## Stage 7 — Tune the strongest tree-based models

In [ ]:
def get_tuning_search_spaces():
    spaces = {
        "Random Forest": {
            "model__n_estimators": [200, 300, 400],
            "model__max_depth": [6, 8, 10, None],
            "model__min_samples_leaf": [1, 2, 4],
        },
    }
    if HAS_XGBOOST:
        spaces["XGBoost"] = {
            "model__n_estimators": [200, 300, 400],
            "model__max_depth": [3, 4, 5, 6],
            "model__learning_rate": [0.03, 0.05, 0.1],
        }
    if HAS_CATBOOST:
        spaces["CatBoost"] = {
            "model__depth": [4, 6, 8],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__iterations": [200, 300, 400],
        }
    return spaces

search_spaces = get_tuning_search_spaces()
tuned_pipelines = {}
tuned_results = []
tuned_cv_results = []

for name, param_grid in search_spaces.items():
    base_model = models[name]
    pipeline = Pipeline(steps=[
        ("preprocessor", build_preprocessor(numerical_features, categorical_features)),
        ("model", base_model),
    ])
    search = RandomizedSearchCV(
        pipeline, param_distributions=param_grid,
        n_iter=15, cv=5, scoring="roc_auc",
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    tuned_pipelines[name] = search.best_estimator_
    tuned_cv_results.append({
        "model": name,
        "cv_roc_auc": search.best_score_,
    })
    print(name, "best CV ROC-AUC:", search.best_score_)

    y_pred = search.best_estimator_.predict(X_test)
    y_proba = search.best_estimator_.predict_proba(X_test)[:, 1]
    tuned_results.append(evaluate_model(f"{name} (tuned)", y_test, y_pred, y_proba))

tuned_comparison = pd.DataFrame(tuned_results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
tuned_comparison

## Stage 8 — Pick the final model and save it

In [ ]:
tuned_cv_table = pd.DataFrame(tuned_cv_results).sort_values("cv_roc_auc", ascending=False).reset_index(drop=True)
tuned_cv_table

# Select the final model using cross-validation on the training data only.
# The test set is kept separate for final evaluation and is not used for model selection.
best_model_name = tuned_cv_table.iloc[0]["model"]
final_pipeline = tuned_pipelines[best_model_name]
print("Final model:", best_model_name)

joblib.dump(final_pipeline, MODEL_PATH)
print(f"Saved model to: {MODEL_PATH}")

## Stage 9 — Build the customer risk table

In [ ]:
def risk_level(probability: float) -> str:
    if probability >= 0.6:
        return "High"
    if probability >= 0.3:
        return "Medium"
    return "Low"

final_proba = final_pipeline.predict_proba(X_test)[:, 1]
final_pred = final_pipeline.predict(X_test)

risk_table = pd.DataFrame({
    "customerID": df.loc[X_test.index, "customerID"].values
    if "customerID" in df.columns else X_test.index,
    "actual_churn": y_test.values,
    "predicted_churn": final_pred,
    "churn_probability": final_proba,
})
risk_table["risk_level"] = risk_table["churn_probability"].apply(risk_level)
risk_table.to_csv(RISK_TABLE_PATH, index=False)
print(f"Saved risk table to: {RISK_TABLE_PATH}")

risk_table.sort_values("churn_probability", ascending=False).head(10)

## Done

The notebook has now produced:
- `data/processed/cleaned_telco_churn.csv`
- `outputs/models/final_churn_model.joblib`
- `outputs/reports/customer_risk_table.csv`

`main.py` in the project root can now load the saved model and score
a single customer, the same way it did before.